# 02 · Demand forecasting

**Goal.** Forecast daily demand per store-item 1–14 days ahead, and measure how much better than a naive rule
we can do.

| Model | Idea |
|---|---|
| Seasonal naive (baseline) | "like last week": the same weekday of the last observed week |
| Moving average (28 days) | flat average of the last four weeks |
| Holt-Winters | level + damped trend + weekly season, one model per series (statsmodels) |
| LightGBM | one global gradient-boosting model on all series with calendar, lag and rolling features |

**Back-test design (no leakage).** A new forecast every 7 days (the *origin*) for 1–14 days ahead. Models are
chosen on the **validation** period and scored on the **test** period. Every LightGBM feature uses data at least
14 days old, so one model serves every horizon without looking into the future (`tests/test_forecast.py` proves
that changing data after the origin never changes the forecast).

**Metrics.** WAPE = Σ|A − F| / ΣA (main), MAPE on days with sales, bias = Σ(F − A) / ΣA, RMSE.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import joblib
import numpy as np
import pandas as pd
from IPython.display import Image, Markdown, display

from src.config import load_config

cfg = load_config(ROOT / "config" / "params.yaml")
RESULTS = ROOT / "data" / "processed" / "work" / "results.joblib"
FIG = ROOT / "reports" / "figures"
assert RESULTS.exists(), "Run `python -m src.pipeline --all` from the project root first (about 2 minutes)."
res = joblib.load(RESULTS)
pd.set_option("display.float_format", "{:,.3f}".format)
pd.set_option("display.max_columns", 30)
print(f"Data: {res.data.file_name} · {res.dm.n_series} series · test period {res.profile.test[0]} → {res.profile.test[1]}")

In [ ]:
bt_v, bt_t = res.backtests["validation"], res.backtests["test"]
print(f"Validation: {res.profile.valid[0]} → {res.profile.valid[1]} · {len(bt_v.origins)} origins")
print(f"Test:       {res.profile.test[0]} → {res.profile.test[1]} · {len(bt_t.origins)} origins")
pd.DataFrame(bt_t.info).T

## 1 · Leaderboard

In [ ]:
lb = res.leaderboard.pivot(index="model", columns="period", values=["wape", "mape", "bias"])
lb.columns = [f"{m} ({p})" for m, p in lb.columns]
lb = lb.sort_values("wape (test)")
lb["WAPE vs baseline (test)"] = 1 - lb["wape (test)"] / lb.loc["seasonal_naive", "wape (test)"]
lb

In [ ]:
display(Image(FIG / "forecast_wape_by_model.png"))
print("Best model on validation:", res.best_model)

## 2 · Where the error comes from

In [ ]:
from src.metrics import evaluate

test_err = res.errors[res.errors["period"] == "test"]
by_h = evaluate(test_err, ["model", "h"]).pivot(index="h", columns="model", values="wape")
display(by_h.style.format("{:.1%}"))
display(Image(FIG / "forecast_wape_by_horizon.png"))

In [ ]:
e = test_err.merge(res.classification[["item", "abc", "xyz"]], on="item")
display(evaluate(e, ["abc", "model"]).pivot(index="abc", columns="model", values="wape").style.format("{:.1%}"))

## 3 · One series, and what LightGBM looks at

In [ ]:
display(Image(FIG / "forecast_example.png"))
if (FIG / "lightgbm_feature_importance.png").exists():
    display(Image(FIG / "lightgbm_feature_importance.png"))

## 4 · Inside Holt-Winters

Parameters estimated with statsmodels for one series. `smoothing_*` close to 0 means a slowly moving
component; `damping_trend` below 1 flattens the trend over the horizon.

In [ ]:
from src.forecast import HoltWinters

store, item = res.dm.keys.iloc[0]
row = res.dm.series_index(store, item)
hw = HoltWinters(n_jobs=1).fit(res.dm.subset([row]), res.dm.date_index(res.profile.valid[1]))
names = ["alpha (level)", "beta (trend)", "gamma (season)", "phi (damping)", "initial level", "initial trend"]
pd.Series(hw.params_[0][:6], index=names, name=f"store {store}, item {item}").round(4)

## 5 · Which series are hardest to forecast?

In [ ]:
best_err = test_err[test_err["model"] == res.best_model]
per_series = evaluate(best_err, ["store", "item"]).merge(res.classification[["item", "weekly_cv", "abc"]], on="item")
display(per_series.sort_values("wape", ascending=False).head(5)[["store", "item", "abc", "weekly_cv", "wape", "bias"]])
volume = best_err.groupby(["store", "item"])["actual"].sum().rename("volume").reset_index()
per_series = per_series.merge(volume, on=["store", "item"])
print(f"Correlation between series volume and WAPE: {np.corrcoef(per_series['volume'], per_series['wape'])[0, 1]:+.2f}")

## 6 · Findings

In [ ]:
h = res.headline["forecast"]
display(Markdown(f"""
* **{h['best_model_label']}** has the lowest error: WAPE **{h['wape_best']:.1%}** on the test period vs
  **{h['wape_baseline']:.1%}** for the seasonal-naive baseline → **{h['wape_improvement_pct']:.0%} less error**.
* Its bias is **{h['bias_best']:+.1%}**: forecasts are neither systematically too high nor too low, which matters
  because the reorder point adds them up over the lead time.
* Low-volume series are the hardest in relative terms (Poisson-like noise weighs more when sales are small);
  that residual noise is what the safety stock has to cover.
* The forecast paths of this model feed the forecast-driven reorder policy in notebook 03.
"""))